# **Projet — Stress hydrique dans les forêts du Valais Central**

<i>Basile Meichtry, d'après le cours de Géoinformatique II, UNIL</i>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<i>10 novembre 2026</i>

Ce notebook a pour objectif de documenter le projet, en présentant la méthode employée et les résultats obtenus.  
<br><br>
### **Question de recherche**

Peut-on prédire le stress hydrique des surfaces forestières du Valais central en 2018 à partir du relief et des caractéristiques de l'urbanisation environnante, sans utiliser de données climatiques ?

### **Contexte**
Le Valais central est particulièrement exposé à la sécheresse en raison de son climat intra-alpin. Le stress hydrique chez les arbres peut limiter leur activité photosynthétique. Le rapport entre évapotranspiration réelle et évapotranspiration potentielle (ET/ETp) permet d'évaluer cette contrainte: une valeur inférieure à 0.8 peut indiquer des conditions de sécheresse affectant la végétation.

Plusieurs facteurs pourraient influencer la répartition spatiale de cet indice. L'urbanisation, notamment par l'imperméabilisation des sols et l'effet d'îlot de chaleur urbain, pourrait modifier la disponibilité en eau pour les forêts. Le relief, à travers l'altitude et la pente, peut également influencer les conditions locales d'exposition et de disponibilité en eau et sera pris en compte dans l'analyse.

Il est donc pertinent d'évaluer dans quelle mesure les caractéristiques topographiques et urbaines permettent de prédire les variations spatiales du stress hydrique forestier, sans intégrer directement de variables climatiques au modèle, afin d'identifier les secteurs potentiellement vulnérables.

### **Hypothèses**

1. Les caractéristiques topographiques expliquent une partie de la variabilité spatiale du stress hydrique.
2. L'ajout de variables liées à l'urbanisation améliore les performances prédictives du modèle fondé uniquement sur le relief.
3. Le relief et l'urbanisation ne suffisent pas à expliquer toute la variabilité observée, en raison de l'absence de variables climatiques.

### **Zone d'étude**
La zone d'étude comprend les districts de Conthey, Sion, Sierre et  Hérens situés dans le canton du Valais, en Suisse.  Le choix de cette zone d'étude permet de considérer une région présentant une diversité de conditions topographiques et environnementales. Cette restriction vise également à réduire les temps de calcul, certaines opérations spatiales sur les couches vectorielles s'avérant particulièrement coûteuses en ressources informatiques.

L'analyse portera sur les surfaces forestières, dont la délimitation repose sur une couche de polygones forestiers datant de 2021. L'indice de stress hydrique étudié correspond quant à lui à l'année 2018. Le décalage temporel entre ces données représente une limite méthodologique. En effet, certaines surfaces forestières ont pu évoluer entre 2018 et 2021. En l'absence d'une couche forestière correspondant à 2018, les données de 2021 seront utilisées comme approximation de la distribution spatiale des forêts durant la période étudiée.

### **Variables**
- **Variable à expliquer** : Indice ET/ETp - rapport entre l'évapotranspiration réelle et l'évapotranspiration potentielle pour l'année 2018. Cette variable représente l'indice de stress hydrique étudié.

- **Variables explicatives** : Les variables explicatives sont réparties en deux groupes.<u><br>Caractéristiques topographiques :</u>
Altitude moyenne du pixel (m) ; Pente moyenne du pixel (°).<u><br>Caractéristiques de l'urbanisation :</u>
Densité urbaine (%) : proportion de la surface du pixel occupée par les zones urbanisées ;
Distance aux zones urbanisées (m) : distance entre le centroïde du pixel forestier et la zone urbanisée la plus proche.<br>

### **Données**
Les données sources sont réunies dans le dossier zip `Meichtry_Basile_projet.zip` téléchargeable via le lien situé tout en haut de la page. Ce dossier contient une geodatabase unique `Meichtry_Basile_projet.gpkg` pour les couches vectorielles, ainsi que les fichier rasters en GeoTIFF. Les couches rasters et vectorielles sont nommées avec un préfixe, selon leur rôle: `source_` pour les données sources intactes, `temp_` pour les résultats intermédiaires, `input_` pour les données résultant de la chaîne de pré-traitements et utilisées comme entrée de la régression, `results_` pour les données finales résultant de l'analyse et utilisées pour l'habillage des cartes.<br>
Toutes les données sont exprimées dans le système de coordonnées LV95 (CH1903+ / LV95, EPSG:2056) ; les altitudes, lorsqu'elles existent, sont référencées au LN02 (EPSG:5728).
<br><br>

| Données utilisées | Nom de la couche | Fournisseur | Type | Lien de téléchargement | Date de téléchargement | Système de coordonnées | Résolution / échelle | Rôle dans l'analyse |
|---|---|---|---|---|---|---|---|---|
|Indice ET/ETp 2018 (`source_Indice_ETETp`)|wald-wasserverfuegbarkeit_pflanzen_2018|Federal Office for the Environment FOEN|Raster|[data.geo.admin](https://data.geo.admin.ch/ch.bafu.wald-wasserverfuegbarkeit_pflanzen/wald-wasserverfuegbarkeit_pflanzen_2018/wald-wasserverfuegbarkeit_pflanzen_2018_2056.tif)|27.10.2026|EPSG:2056 - CH1903+ / LV95|250x250| variable à expliquer
|Modèle Numérique de Terrain (`source_MNT_10x10m`)| swissALTIRegio| swisstopo| Raster|[swisstopo]( https://www.swisstopo.admin.ch/fr/modele-altimetrique-swissaltiregio) |01.11.2026|EPSG:2056 - CH1903+ / LV95| 10x10m| variable d’entrée pour le calcul de l’altitude et de la pente
|Couverture du sol 2021 (`source_Couverture_du_sol`)| swissTLM3D_2021 (TLM_BODENBEDECKUNG)| swisstopo|vecteur (polygones)|[swisstopo](https://www.swisstopo.admin.ch/de/landschaftsmodell-swisstlm3d)| 27.10.2026| LV95 LN02| 1-3m|variable d’entrée et délimitation des surfaces forestières|
|Routes 2021 (`source_Routes`)| swissTLM3D_2021 (TLM_STRASSE)| swisstopo| vecteur (lignes)| [swisstopo](https://www.swisstopo.admin.ch/de/landschaftsmodell-swisstlm3d)| 27.10.2026|LV95 LN02| 0.2-1.5m| variable d’entrée pour calcul de distance et de densité|
|Bâtiments 2021 (`source_Batiments`)| swissTLM3D_2021 (TLM_GEBAEUDE_FOOTPRINT)| swisstopo| vecteur (polygones)|[swisstopo](https://www.swisstopo.admin.ch/de/landschaftsmodell-swisstlm3d) |27.10.2026| LV95 LN02| 0.2-1.5m| variable d’entrée pour calcul de distance et de densité|
|Limites des districts (`source_limite_districts`) |swissBOUNDARIES3D (TLM_BEZIRKSGEBIET)| swisstopo| vecteur (polygones)|[swisstopo]( https://www.swisstopo.admin.ch/fr/modele-du-territoire-swissboundaries3d)| 27.10.2026|LV95 LN02| ± 0,5 m|Délimitation de la zone d’étude|

<br><br>Les couches et rasters `source_` restent intacts. Les couche préfixées `temp_`  et `input_` ont été réalisées à l'aide de QGIS par le biais du modeleur graphique. Le fichier correspondant, `Stress_hydrique.model3`est également disponible dans le fichier zip `Meichtry_Basile_projet.zip`.<br> Ce notebook ajoute à la geodatabase les couches `results_`.

### Reproduire ce notebook

1. **Données** : Téléchargez `Meichtry_Basile_projet.zip` depuis le haut de cette page et décompressez-le pour obtenir le dossier `Meichtry_Basile_projet/`. Les chemins d'accès sont relatifs et définis dans les cellules de configuration (§1). Il est important de conserver l'organisation des dossiers.

2. **Environnement** : Ce notebook a été développé dans **Google Colab**.
Les versions des six bibliothèques utilisées (`numpy, pandas, rasterio, geopandas, matplotlib et scikit-learn`) sont indiquées dans le fichier `requirements.txt`. La version de Python utilisée est également précisée dans le fichier. Pour installer les mêmes versions de bibliothèques, exécutez la commande suivante dans un terminal ouvert dans le dossier du projet: `pip install -r requirements.txt`.<br>
La version de Python doit être compatible avec les versions des bibliothèques indiquées dans le fichier.

3. **Exécution** : Ouvrez le notebook et exécutez les cellules dans l'ordre, à l'aide de l'option « Exécuter toutes les cellules » (Run all).
La graine aléatoire est fixée dans la cellule de configuration (§1).

## Plan

1. Configuration · 2. Données sources · 3. Analyse exploratoire · 4. Chaîne de pré-traitements · 5. Modélisation et choix du modèle · 6. Analyse des résidus · 7. Carte des prédictions et incertitude · 8. Export des résultats · 9. Conclusion








---
## 1. Configuration

In [23]:
# Bibliothèques
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import sklearn

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Chemins relatifs au dossier du notebook
PROJET = Path("Meichtry_Basile_projet")

GPKG = PROJET / "Meichtry_Basile.gpkg"
RASTER_ET_ETP = PROJET / "source_Indice_ETETp.tif"
RASTER_MNT = PROJET / "source_MNT_10x10m.tif"

# Graine aléatoire
SEED = 21

# Affichage des versions utilisées
print(
    f"Exécution du notebook le {date.today():%d.%m.%Y}\n"
    "Versions :\n"
    f"Python {sys.version.split()[0]}, "
    f"numpy {np.__version__}, "
    f"pandas {pd.__version__}, "
    f"geopandas {gpd.__version__}, "
    f"rasterio {rasterio.__version__}, "
    f"scikit-learn {sklearn.__version__}"
)


Exécution du notebook le 10.11.2026
Versions :
Python 3.13.18, numpy 2.4.6, pandas 2.2.3, geopandas 1.2.0, rasterio 1.5.2, scikit-learn 1.6.1


---
## 2. Données sources

---
## 3. Analyse exploratoire des données brutes

---
## 4. Chaîne de pré-traitements : des données sources au tableau d'analyse

---
## 5. Modélisation : ajustement, évaluation et choix du modèle

---
## 6. Analyse des résidus

---
## 7. Carte des prédictions et incertitude

---
## 8. Export des résultats

---
## 9. Conclusion